# ⚽ Football Tactical Analyzer — Final Version
- 🟢 Team A: AUTUMN colormap (black→orange→yellow)
- 🔵 Team B: WINTER colormap (blue→cyan)
- ⚡ Speed: computed every 10 frames only, displayed as stable value
- 🧩 Formation: only shown when 8+ players visible, majority-voted
- 🗺️ Radar: bird's-eye view bottom-right

In [ ]:
!pip install ultralytics supervision opencv-python-headless numpy scikit-learn filterpy -q
print('✅ Done!')

In [ ]:
from google.colab import files
print('📂 Upload your football video...')
uploaded    = files.upload()
VIDEO_PATH  = list(uploaded.keys())[0]
OUTPUT_PATH = 'football_output.mp4'
print('✅ Uploaded: ' + VIDEO_PATH)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━ CONFIG ━━━━━━━━━━━━━━━━━━━
CONF_THRESHOLD        = 0.45
IMG_SIZE              = 1088

# Heatmap
HEATMAP_ALPHA         = 0.50   # overlay strength (0=invisible, 1=full)
BLOB_RADIUS           = 18     # gaussian blob radius
HEATMAP_THRESHOLD     = 8      # ignore pixels below this intensity (keeps pitch clean)

# Speed — computed every N frames so display is stable, not flickery
SPEED_COMPUTE_EVERY   = 10     # only recompute speed every 10 frames
SPEED_HISTORY_LEN     = 6      # average over last 6 readings (~60 frames)
FIELD_LENGTH_M        = 105.0
MAX_SPEED_KMH         = 35.0   # hard cap, above this = tracker glitch

# Formation
MIN_PLAYERS_FORMATION = 8      # don't guess formation with fewer players
FORMATION_EVERY_N     = 20     # compute every N frames
FORMATION_VOTE_LEN    = 40     # majority vote over last N snapshots

# Classes
PERSON_CLASS = 0
BALL_CLASS   = 32

# Radar
RADAR_W, RADAR_H = 190, 120

# Team box colors (BGR)
COLOR_A = (30,  200,  30)   # green  — Team A boxes
COLOR_B = (220, 180,   0)   # cyan/blue — Team B boxes

print('✅ Config set!')

In [ ]:
from ultralytics import YOLO
import supervision as sv

model   = YOLO('yolo26n.pt')
tracker = sv.ByteTrack()
print('✅ YOLO26 + ByteTrack loaded!')

## 🎨 Team Calibration (LAB color space)

In [ ]:
import cv2
import numpy as np
from sklearn.cluster import KMeans


def get_jersey_color(frame, box):
    """Extract dominant jersey color from torso region in LAB space."""
    x1, y1, x2, y2 = [int(v) for v in box]
    h    = y2 - y1
    crop = frame[y1 : y1 + int(h * 0.38), x1:x2]
    if crop.size == 0 or crop.shape[0] < 3 or crop.shape[1] < 3:
        return np.array([50.0, 0.0, 0.0])
    # Mask grass
    hsv   = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV).reshape(-1,3).astype(np.float32)
    grass = (hsv[:,0]>35) & (hsv[:,0]<85) & (hsv[:,1]>40)
    lab   = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB).reshape(-1,3).astype(np.float64)
    px    = lab[~grass]
    if len(px) < 15:
        return np.array([50.0, 0.0, 0.0])
    km     = KMeans(n_clusters=2, n_init=3, random_state=0).fit(px)
    counts = np.bincount(km.labels_)
    return km.cluster_centers_[np.argmax(counts)].astype(np.float64)


def calibrate_teams(video_path, mdl, n_frames=40):
    cap    = cv2.VideoCapture(video_path)
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    idxs   = np.linspace(10, total-10, n_frames, dtype=int)
    colors = []
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frm = cap.read()
        if not ret:
            continue
        res = mdl(frm, verbose=False, conf=0.45, imgsz=640, classes=[PERSON_CLASS])[0]
        for box in res.boxes:
            colors.append(get_jersey_color(frm, box.xyxy[0].cpu().numpy()))
    cap.release()
    if len(colors) < 6:
        print('⚠ Not enough players — team split skipped')
        return None
    arr = np.array(colors, dtype=np.float64)
    km  = KMeans(n_clusters=2, n_init=15, random_state=42).fit(arr)
    print('✅ Teams calibrated!')
    print('   Team A LAB: ' + str(km.cluster_centers_[0].astype(int)))
    print('   Team B LAB: ' + str(km.cluster_centers_[1].astype(int)))
    return km


print('🔍 Calibrating teams (~20s)...')
team_km = calibrate_teams(VIDEO_PATH, model, n_frames=40)

## 🔧 Helpers

In [ ]:
from filterpy.kalman import KalmanFilter
from collections import defaultdict, Counter


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# SPEED — key insight:
#   compute displacement only every SPEED_COMPUTE_EVERY frames
#   this gives a MUCH more stable reading because:
#   - tiny 1-2px jitter per frame = 0 km/h over 10 frames
#   - real movement = consistent positive displacement over 10 frames
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
speed_anchor     = {}   # tid -> (cx,cy) at last measurement frame
speed_anchor_frm = {}   # tid -> frame_idx of last measurement
speed_history    = defaultdict(list)  # tid -> list of computed km/h
speed_display    = {}   # tid -> stable displayed km/h
speed_log_all    = []   # for chart

def compute_speed(tid, cx, cy, frame_idx, fw, fps):
    """
    Only recomputes every SPEED_COMPUTE_EVERY frames.
    Between measurements, returns the last stable value.
    """
    last_frm = speed_anchor_frm.get(tid, -999)

    if frame_idx - last_frm >= SPEED_COMPUTE_EVERY:
        if tid in speed_anchor:
            px, py   = speed_anchor[tid]
            frames_elapsed = frame_idx - last_frm
            dist_px  = np.sqrt((cx-px)**2 + (cy-py)**2)
            mpp      = FIELD_LENGTH_M / fw
            # km/h = (dist_m / time_s) * 3.6
            kmh      = dist_px * mpp / (frames_elapsed / fps) * 3.6
            if kmh <= MAX_SPEED_KMH:   # reject tracker jumps
                speed_history[tid].append(kmh)
                if len(speed_history[tid]) > SPEED_HISTORY_LEN:
                    speed_history[tid].pop(0)
                speed_display[tid] = round(float(np.mean(speed_history[tid])), 1)
        speed_anchor[tid]     = (cx, cy)
        speed_anchor_frm[tid] = frame_idx

    return speed_display.get(tid, 0.0)


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# GAUSSIAN BLOB
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def add_blob(heatmap, cx, cy, radius=BLOB_RADIUS):
    h, w  = heatmap.shape
    r     = radius * 2
    y0,y1 = max(0,cy-r), min(h,cy+r+1)
    x0,x1 = max(0,cx-r), min(w,cx+r+1)
    if y0>=y1 or x0>=x1:
        return
    yy,xx = np.mgrid[y0:y1, x0:x1]
    heatmap[y0:y1, x0:x1] += np.exp(-((xx-cx)**2+(yy-cy)**2)/(2.0*radius**2))


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# FORMATION — majority vote over window
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
form_buf_A = []
form_buf_B = []
formation_A_history = []
formation_B_history = []

def detect_formation(positions):
    if len(positions) < MIN_PLAYERS_FORMATION:
        return None
    pts      = np.array(positions)
    pts      = pts[np.argsort(pts[:,0])]
    outfield = pts[1:]  # drop GK
    if len(outfield) < 5:
        return None
    n_lines  = min(4, len(outfield))
    km = KMeans(n_clusters=n_lines, n_init=5, random_state=0)
    km.fit(outfield[:,0].reshape(-1,1))
    order  = np.argsort(km.cluster_centers_[:,0])
    counts = [int(np.sum(km.labels_==o)) for o in order]
    return '-'.join(str(c) for c in counts)

def majority_vote(buf):
    if not buf:
        return '?'
    return Counter(buf[-FORMATION_VOTE_LEN:]).most_common(1)[0][0]


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# RADAR
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def draw_radar(player_data, fw, fh):
    r = np.zeros((RADAR_H, RADAR_W, 3), dtype=np.uint8)
    cv2.rectangle(r,(0,0),(RADAR_W-1,RADAR_H-1),(34,105,34),-1)
    cv2.line(r,(RADAR_W//2,0),(RADAR_W//2,RADAR_H),(60,140,60),1)
    cv2.ellipse(r,(RADAR_W//2,RADAR_H//2),(22,18),0,0,360,(60,140,60),1)
    cv2.rectangle(r,(2,2),(RADAR_W-3,RADAR_H-3),(60,140,60),1)
    for (tid, team_id, box) in player_data:
        x1,y1,x2,y2 = box
        rx = max(4, min(RADAR_W-5, int((x1+x2)/2/fw*RADAR_W)))
        ry = max(4, min(RADAR_H-5, int((y1+y2)/2/fh*RADAR_H)))
        col = COLOR_A if team_id==0 else COLOR_B
        cv2.circle(r,(rx,ry),4,col,-1)
        cv2.circle(r,(rx,ry),4,(200,200,200),1)
    return r


print('✅ All helpers ready!')

## 🎬 Main Processing Loop

In [ ]:
cap    = cv2.VideoCapture(VIDEO_PATH)
fw     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
fh     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS) or 25.0
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (fw, fh))

heatmap_A = np.zeros((fh, fw), dtype=np.float32)
heatmap_B = np.zeros((fh, fw), dtype=np.float32)

frame_idx      = 0
track_team_map = {}

print('🎬 Processing... (Runtime → Change runtime type → T4 GPU for 3x speed)')

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame_idx += 1
    annotated  = frame.copy()

    # ── YOLO26 ───────────────────────────────────────────────────────────────
    results    = model(frame, verbose=False, conf=CONF_THRESHOLD,
                       imgsz=IMG_SIZE, classes=[PERSON_CLASS, BALL_CLASS])[0]
    detections = sv.Detections.from_ultralytics(results)

    # ── Ball ─────────────────────────────────────────────────────────────────
    ball_mask = detections.class_id == BALL_CLASS
    if ball_mask.any():
        bx1,by1,bx2,by2 = detections.xyxy[ball_mask][0].astype(int)
        bcx,bcy = (bx1+bx2)//2,(by1+by2)//2
        cv2.circle(annotated,(bcx,bcy),10,(0,255,255),-1)
        cv2.circle(annotated,(bcx,bcy),10,(0,0,0),2)

    # ── ByteTrack ─────────────────────────────────────────────────────────────
    person_dets = detections[detections.class_id == PERSON_CLASS]
    person_dets = tracker.update_with_detections(person_dets)

    player_data = []
    positions_A = []
    positions_B = []

    for xyxy, tid in zip(person_dets.xyxy, person_dets.tracker_id):
        if tid is None:
            continue
        x1,y1,x2,y2 = xyxy.astype(int)
        cx = (x1+x2)//2
        cy = y2   # foot

        # ── Team assignment (sticky) ────────────────────────────────────────
        if tid not in track_team_map:
            if team_km is not None:
                col     = get_jersey_color(frame,(x1,y1,x2,y2))
                team_id = int(team_km.predict([col.astype(np.float64)])[0])
            else:
                team_id = 0
            track_team_map[tid] = team_id
        team_id = track_team_map[tid]

        # ── Speed (stable: computed every N frames only) ─────────────────────
        kmh = compute_speed(tid, cx, cy, frame_idx, fw, fps)
        if frame_idx % SPEED_COMPUTE_EVERY == 0:
            speed_log_all.append((frame_idx, int(tid), kmh))

        # ── Draw box + stable speed label ───────────────────────────────────
        col = COLOR_A if team_id==0 else COLOR_B
        cv2.rectangle(annotated,(x1,y1),(x2,y2),col,2)
        cv2.putText(annotated, str(kmh)+' km/h', (x1, y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.40, col, 1)

        # ── Heatmap blob ────────────────────────────────────────────────────
        if 0 <= cy < fh and 0 <= cx < fw:
            if team_id == 0:
                add_blob(heatmap_A, cx, cy)
                positions_A.append([cx, cy])
            else:
                add_blob(heatmap_B, cx, cy)
                positions_B.append([cx, cy])

        player_data.append((tid, team_id, (x1,y1,x2,y2)))

    # ── Formation snapshot ──────────────────────────────────────────────────
    if frame_idx % FORMATION_EVERY_N == 0:
        fa = detect_formation(positions_A)
        fb = detect_formation(positions_B)
        if fa:
            form_buf_A.append(fa)
            formation_A_history.append((frame_idx, fa))
        if fb:
            form_buf_B.append(fb)
            formation_B_history.append((frame_idx, fb))

    cur_fa = majority_vote(form_buf_A)
    cur_fb = majority_vote(form_buf_B)

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # HEATMAP OVERLAY
    # Strategy:
    #   - Normalize each heatmap 0→255
    #   - Apply AUTUMN colormap for Team A (orange/yellow tones)
    #   - Apply WINTER colormap for Team B (blue/cyan tones)
    #   - Zero out pixels below HEATMAP_THRESHOLD so pitch stays clean
    #   - Blend only where signal exists
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    hm_out = np.zeros((fh, fw, 3), dtype=np.float32)

    if heatmap_A.max() > 0:
        normA  = cv2.normalize(heatmap_A, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        hmA    = cv2.applyColorMap(normA, cv2.COLORMAP_AUTUMN).astype(np.float32)
        maskA  = normA > HEATMAP_THRESHOLD
        hm_out[maskA] += hmA[maskA]

    if heatmap_B.max() > 0:
        normB  = cv2.normalize(heatmap_B, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        hmB    = cv2.applyColorMap(normB, cv2.COLORMAP_WINTER).astype(np.float32)
        maskB  = normB > HEATMAP_THRESHOLD
        # Where B overlaps A: blend 50/50 instead of overwrite
        both   = maskB & (hm_out.sum(axis=2) > 0)
        only_b = maskB & ~both
        hm_out[only_b] += hmB[only_b]
        hm_out[both]    = hm_out[both] * 0.5 + hmB[both] * 0.5

    hm_out = np.clip(hm_out, 0, 255).astype(np.uint8)

    # Blend onto video only where heatmap has signal
    signal = hm_out.sum(axis=2) > 0
    annotated[signal] = cv2.addWeighted(
        annotated, 1.0 - HEATMAP_ALPHA,
        hm_out,    HEATMAP_ALPHA, 0
    )[signal]

    # ── Radar (bottom-right) ─────────────────────────────────────────────────
    radar  = draw_radar(player_data, fw, fh)
    ry_off = fh - RADAR_H - 32
    rx_off = fw - RADAR_W - 10
    annotated[ry_off:ry_off+RADAR_H, rx_off:rx_off+RADAR_W] = radar

    # ── Formation (top-right) ────────────────────────────────────────────────
    cv2.rectangle(annotated,(fw-195,4),(fw-4,58),(0,0,0),-1)
    cv2.rectangle(annotated,(fw-195,4),(fw-4,58),(70,70,70),1)
    cv2.putText(annotated,'A: '+cur_fa,(fw-185,28),
                cv2.FONT_HERSHEY_SIMPLEX,0.65,COLOR_A,2)
    cv2.putText(annotated,'B: '+cur_fb,(fw-185,52),
                cv2.FONT_HERSHEY_SIMPLEX,0.65,COLOR_B,2)

    # ── HUD bar (bottom) ─────────────────────────────────────────────────────
    n_a = sum(1 for d in player_data if d[1]==0)
    n_b = sum(1 for d in player_data if d[1]==1)
    hud = 'Frame '+str(frame_idx)+'  |  Team A: '+str(n_a)+'  |  Team B: '+str(n_b)
    cv2.rectangle(annotated,(0,fh-28),(fw,fh),(0,0,0),-1)
    cv2.putText(annotated,hud,(8,fh-8),cv2.FONT_HERSHEY_SIMPLEX,0.50,(255,255,255),1)

    # ── Legend (top-left) ─────────────────────────────────────────────────────
    cv2.rectangle(annotated,(4,4),(175,56),(0,0,0),-1)
    cv2.rectangle(annotated,(4,4),(175,56),(70,70,70),1)
    cv2.putText(annotated,'Team A',(12,26),cv2.FONT_HERSHEY_SIMPLEX,0.52,COLOR_A,1)
    cv2.putText(annotated,'Team B',(12,49),cv2.FONT_HERSHEY_SIMPLEX,0.52,COLOR_B,1)

    out.write(annotated)

    if frame_idx % 50 == 0:
        print('  ✔ Frame '+str(frame_idx)+' | A:'+str(n_a)+' B:'+str(n_b)
              +' | '+cur_fa+' vs '+cur_fb)

cap.release()
out.release()
print('\n✅ Done! → '+OUTPUT_PATH)

## 📊 Analytics Report

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import pandas as pd

fig = plt.figure(figsize=(20,12))
gs  = gridspec.GridSpec(2,3,figure=fig,hspace=0.4,wspace=0.35)

ax1 = fig.add_subplot(gs[0,0])
ax1.imshow(heatmap_A, cmap='autumn', aspect='auto')
ax1.set_title('Team A — Heatmap',fontsize=13,fontweight='bold',color='#dd6600')
ax1.axis('off')

ax2 = fig.add_subplot(gs[0,1])
ax2.imshow(heatmap_B, cmap='winter', aspect='auto')
ax2.set_title('Team B — Heatmap',fontsize=13,fontweight='bold',color='#0055cc')
ax2.axis('off')

ax3 = fig.add_subplot(gs[0,2])
rgb = np.zeros((fh,fw,3),dtype=np.float32)
if heatmap_A.max()>0:
    rgb[:,:,0] += cv2.normalize(heatmap_A,None,0,1,cv2.NORM_MINMAX)
    rgb[:,:,1] += cv2.normalize(heatmap_A,None,0,0.4,cv2.NORM_MINMAX)
if heatmap_B.max()>0:
    rgb[:,:,2] += cv2.normalize(heatmap_B,None,0,1,cv2.NORM_MINMAX)
ax3.imshow(np.clip(rgb,0,1),aspect='auto')
ax3.set_title('Combined',fontsize=13,fontweight='bold')
ax3.axis('off')

ax4 = fig.add_subplot(gs[1,0])
if speed_log_all:
    df = pd.DataFrame(speed_log_all,columns=['frame','tid','kmh'])
    df['team'] = df['tid'].map(lambda t: track_team_map.get(t,0))
    bins = np.linspace(0,MAX_SPEED_KMH,20)
    ax4.hist(df[df['team']==0]['kmh'].values,bins=bins,alpha=0.75,color='#dd6600',label='Team A')
    ax4.hist(df[df['team']==1]['kmh'].values,bins=bins,alpha=0.75,color='#0055cc',label='Team B')
    ax4.set_xlabel('Speed (km/h)'); ax4.set_ylabel('Count')
    ax4.set_title('Speed Distribution',fontsize=13,fontweight='bold')
    ax4.legend(); ax4.grid(alpha=0.3)
else:
    ax4.text(0.5,0.5,'No speed data',ha='center',va='center')

ax5 = fig.add_subplot(gs[1,1])
if speed_log_all:
    mx  = df.groupby('tid')['kmh'].max().reset_index()
    mx['team'] = mx['tid'].map(lambda t: track_team_map.get(t,0))
    mx  = mx.sort_values('kmh',ascending=False).head(14)
    cb  = ['#dd6600' if t==0 else '#0055cc' for t in mx['team']]
    ax5.barh(['P'+str(int(t)) for t in mx['tid']],mx['kmh'],color=cb)
    ax5.set_xlabel('Max Speed (km/h)')
    ax5.set_title('Top Speeds per Player',fontsize=13,fontweight='bold')
    ax5.legend(handles=[mpatches.Patch(color='#dd6600',label='A'),
                         mpatches.Patch(color='#0055cc',label='B')])
    ax5.grid(alpha=0.3,axis='x')
else:
    ax5.text(0.5,0.5,'No speed data',ha='center',va='center')

ax6 = fig.add_subplot(gs[1,2])
if formation_A_history or formation_B_history:
    if formation_A_history:
        ft=[f/fps for f,_ in formation_A_history]
        fl=[l for _,l in formation_A_history]
        ax6.scatter(ft,[1.1]*len(ft),c='#dd6600',s=55,zorder=3)
        for x,l in zip(ft,fl): ax6.text(x,1.14,l,fontsize=7,ha='center',color='#dd6600')
    if formation_B_history:
        ft=[f/fps for f,_ in formation_B_history]
        fl=[l for _,l in formation_B_history]
        ax6.scatter(ft,[0.9]*len(ft),c='#0055cc',s=55,zorder=3)
        for x,l in zip(ft,fl): ax6.text(x,0.86,l,fontsize=7,ha='center',color='#0055cc')
    ax6.set_xlabel('Time (s)')
    ax6.set_yticks([0.9,1.1]); ax6.set_yticklabels(['Team B','Team A'])
    ax6.set_ylim(0.75,1.3)
    ax6.set_title('Formation Timeline',fontsize=13,fontweight='bold')
    ax6.grid(alpha=0.3)
else:
    ax6.text(0.5,0.5,'No formation data\n(clip too short?)',ha='center',va='center')

plt.suptitle('Football Tactical Analysis Report',fontsize=16,fontweight='bold')
plt.savefig('analytics_report.png',dpi=130,bbox_inches='tight')
plt.show()
print('✅ Report saved!')

In [ ]:
from google.colab import files
import os
for f in [OUTPUT_PATH,'analytics_report.png']:
    if os.path.exists(f):
        files.download(f)
        print('✅ Downloaded: '+f)
    else:
        print('⚠ Not found: '+f)